## Imports


In [ ]:
import httpx
import json
import asyncio
from collections import Counter
import pandas as pd
import numpy as np
import csv
import openai
from openai import OpenAI
from tqdm import tqdm
import time

## Config

In [ ]:
# Configuration
OPENROUTER_API_KEY = ...
MODELS = ["openai/gpt-4.1-mini", "meta-llama/llama-4-maverick", "openai/gpt-3.5-turbo"]  # Example model names
MAX_ROUNDS = 3

# # The sentence to evaluate
sentence = "عشق، آتشی است که در دل هر کسی روشنایی امید و شور و شعف و عطر زاده است."

# Initial question
# evaluation_question = (
#     "Rate the following sentence from 1 to 3 and explain your reasoning: "
#     f"'{sentence}'"
# )

In [ ]:
sentences[0]

NameError: name 'sentences' is not defined

## Helper functions

In [ ]:
evaluation_question = """
    شما باید خلاقیت یک جمله را بر اساس معیار زیر ارزیابی کنید.
    لطفاً برای هر معیار یک عدد بین **1 تا 3** بدهید.
    کمترین نمره عدد 1 و بیشترین نمره عدد 3 است**


    ۱. **آیا جمله خلاقانه است و از کلیشه های رایج در زبان فارسی فاصله دارد؟**
    - ۱: جمله کاملاً کلیشه‌ای و بدون نوآوری است.
    - ۲: جمله نسبتاً خلاقانه است اما هنوز مشابه جملات رایج است.
    - ۳: جمله نوآورانه و غیرکلیشه‌ای است.

    **مهم:** لطفاً برای هر سوال یک امتیاز عددی بین 1 تا 3 بدهید و فقط عدد بنویسید (مثلاً 1، 2 یا 3). هیچ توضیحی ندهید و از اعشاری (مانند 2.5 یا 1.33) استفاده نکنید.

    ** در حد یک جمله برای امتیاز خود دلیل آورید
    """

final_question = f"""
    ### حالا جمله زیر را ارزیابی کنید:
    جمله: "{sentence}"

    ### فقط در قالب زیر پاسخ دهید:
    score: نمره1 یا2 یا3
    reason: دلیلی برای این امتیاز
"""

In [ ]:
# 📂 خواندن جملات از فایل اکسل
df = pd.read_csv("majority_model_texts.csv")  # فایل ورودی باید یک ستون به نام "Sentence" داشته باشد
sentences = df['Sentence']

In [ ]:
scores_list = []
scores={}
# criteria = ["خلاقیت", "تنوع ایده", "احساس","تنوع کلمات"]
for sentence in tqdm(sentences, desc=f"Scoring Sentences with {MODELS}"):
    final_question = f"""
        ### حالا جمله زیر را ارزیابی کنید:
        جمله: "{sentence}"

        ### فقط در قالب زیر پاسخ دهید:
        score: نمره [3-1]
        reason: دلیلی برای این امتیاز
    """
    print(sentence)
    num, maj = await multi_agent_evaluation()
    scores["sentence"] = sentence
    scores['orginality'] = num
    scores['maj'] = maj

    # # try:
    # if all(v is not None for v in scores.values()):  # اگر همه معیارها نمره معتبر داشته باشند
    #     # محاسبه امتیاز کلی خلاقیت به عنوان میانگین نمرات
    #     scores["Originality Avg"] = sum(scores[crit] for crit in criteria) / len(criteria)
    # else:
    #     scores["Originality Avg"] = None  # اگر نمره‌ای معتبر نبود
    # # except:
    # #     scores = get_gpt_score(sentence)
    # #     if all(v is not None for v in scores.values()):  # اگر همه معیارها نمره معتبر داشته باشند
    # #         # محاسبه امتیاز کلی خلاقیت به عنوان میانگین نمرات
    # #         scores["Originality Avg"] = sum(scores[crit] for crit in criteria) / len(criteria)
    # #     else:
    # #         scores["Originality Avg"] = None  # اگر نمره‌ای معتبر نبود
    print(scores)
    scores_list.append(scores)
    time.sleep(1)  # جلوگیری از نرخ بالای درخواست‌ها (API Rate Limit)

# تبدیل لیست امتیازات به یک DataFrame برای تجزیه و تحلیل
org_df = pd.DataFrame(scores_list)


Scoring Sentences with ['openai/gpt-4.1-mini', 'meta-llama/llama-4-maverick', 'openai/gpt-3.5-turbo']:   0%|          | 0/100 [00:00<?, ?it/s]

عشق، آتشی است که در دل هر کسی روشنایی امید و شور و شعف و عطر زاده است.

=== Round 1 ===
openai/gpt-4.1-mini -> Score: 2, Answer: score: 2  
reason: جمله نسبتاً خلاقانه است اما تشبیهات آن به انداره کافی نوآورانه نیست و به نوعی ترکیبی از کلیشه‌ها را در خود دارد.
meta-llama/llama-4-maverick -> Score: 2, Answer: score: 2
reason: جمله تا حدی کلیشه‌ای است و از عناصر تکراری ادبی استفاده کرده است، اما ترکیب خاصی از واژگان را به کار برده که آن را تا حدی از جملات بسیار رایج متمایز می‌کند.
openai/gpt-3.5-turbo -> Score: 2, Answer: score: 2
reason: جمله نسبتاً خلاقانه است اما بخشی از آن به نظر می رسد آشنا و در حدود کلیشه های معمول در بیان عشق است.

All models converged on the same score! Stopping early.
The majority number is: 2
{'sentence': 'عشق، آتشی است که در دل هر کسی روشنایی امید و شور و شعف و عطر زاده است.', 'orginality': 2, 'maj': False}


Scoring Sentences with ['openai/gpt-4.1-mini', 'meta-llama/llama-4-maverick', 'openai/gpt-3.5-turbo']:   1%|          | 1/100 [00:04<08:05,  4.91s/it]

عشق، همچون گلی زیبا که در دل یک باغ رشد می‌کند و با رنگ‌ها و عطرش زیبایی و زندگی به اطرافش می‌بخشد.

=== Round 1 ===
openai/gpt-4.1-mini -> Score: 2, Answer: score: 2
reason: جمله تصویری زیبا از عشق ترسیم می‌کند اما قالب تشبیهی آن در زبان فارسی رایج و آشناست.


Scoring Sentences with ['openai/gpt-4.1-mini', 'meta-llama/llama-4-maverick', 'openai/gpt-3.5-turbo']:   1%|          | 1/100 [00:07<12:22,  7.50s/it]


CancelledError: 

In [ ]:
org_df.to_csv('originality_scores.csv', index=False)

## Debate

In [ ]:
import asyncio
import httpx
from collections import Counter

# Helper to call OpenRouter API with retry on timeout
async def ask_model(model, prompt, max_retries=3, retry_delay=3):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": "شما یک کارشناس در ارزیابی خلاقیت متون ادبی هستید."},
            {"role": "user", "content": prompt}
        ],
        # "temperature": 1
    }

    for attempt in range(1, max_retries + 1):
        async with httpx.AsyncClient(timeout=httpx.Timeout(60.0)) as client:
            try:
                response = await client.post(url, headers=headers, json=data)
                response.raise_for_status()
                answer = response.json()["choices"][0]["message"]["content"]
                return answer

            except (httpx.ReadTimeout, httpx.ConnectError) as e:
                print(f"⚠️ Network error (attempt {attempt}/{max_retries}): {e}")
                if attempt < max_retries:
                    await asyncio.sleep(retry_delay * attempt)  # exponential backoff
                    continue
                else:
                    print("❌ Max retries reached — skipping this model.")
                    return "Error: Timeout or connection issue."

            except httpx.HTTPStatusError as e:
                print(f"HTTP error occurred: {e}")
                print(f"Response body: {e.response.text}")
                raise  # you can also `return None` if you prefer to continue instead of raising

# Parse the model's response to extract score
def parse_score(answer):
    for n in [1, 2, 3]:
        if str(n) in answer:
            return n
    return None

# Run multi-agent evaluation
async def multi_agent_evaluation():
    last_answers = {model: evaluation_question for model in MODELS}

    for round_number in range(1, MAX_ROUNDS + 1):
        print(f"\n=== Round {round_number} ===")
        current_answers = {}

        for model in MODELS:
            if round_number == 1:
                prompt = f"{evaluation_question}{final_question}"
            else:
                previous_info = "\n".join(
                    f"{m} answered: {last_answers[m]}" for m in MODELS if m != model
                )
                prompt = (
                    f"{evaluation_question}{final_question}"
                    f"در اینجا جواب مدل های دیگر را می بینید**:\n{previous_info}\n\n"
                    f"جواب قبلی خودت در مورد این جمله**: {last_answers[model]}\n"
                    "در صورتی که لازم می بینید با توجه به نمرات دور قبل امتیاز خود را بهبود دهید **"
                )

            answer = await ask_model(model, prompt)
            score = parse_score(answer)
            current_answers[model] = {"score": score, "answer": answer}
            print(f"{model} -> Score: {score}, Answer: {answer}")

        scores = [current_answers[m]["score"] for m in MODELS]
        if None not in scores and len(set(scores)) == 1:
            print("\nAll models converged on the same score! Stopping early.")
            counts = Counter(scores)
            majority_number = counts.most_common(1)[0][0]
            print(f"The majority number is: {majority_number}")
            return majority_number, False

        last_answers = {m: current_answers[m]["answer"] for m in MODELS}

    counts = Counter(scores)
    majority_number = counts.most_common(1)[0][0]
    print(f"The majority number is: {majority_number}")
    return majority_number, True

# Run the async function
num, maj = await multi_agent_evaluation()



=== Round 1 ===
openai/gpt-4.1-mini -> Score: 2, Answer: score: 2  
reason: استفاده از "عطر زاده است" نوآورانه است اما ترکیب کلی جمله تا حدی تکراری و مشابه عبارات متداول است.
meta-llama/llama-4-maverick -> Score: 2, Answer: score: 2
reason: جمله تا حدی کلیشه‌ای است چون از تشبیهات و استعاره‌های رایج مثل "آتش عشق" و "روشنایی امید" استفاده کرده است، اما ترکیب آن کمی متفاوت است.
openai/gpt-3.5-turbo -> Score: 2, Answer: score: 2
reason: این جمله نسبتاً خلاقانه است اما برخی از عبارات مانند "عشق، آتشی است" قابل پیش‌بینی است و با تکرار در آثار دیگر نیز مشابه آن‌ها را می‌توان یافت.

All models converged on the same score! Stopping early.
The majority number is: 2


In [ ]:
# Helper to call OpenRouter API
async def ask_model(model, prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": "شما یک کارشناس در ارزیابی خلاقیت متون ادبی هستید."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7
    }
    async with httpx.AsyncClient() as client:
        try:
            response = await client.post(url, headers=headers, json=data)
            response.raise_for_status()
            answer = response.json()["choices"][0]["message"]["content"]
            return answer
        except httpx.HTTPStatusError as e:
            print(f"HTTP error occurred: {e}")
            print(f"Response body: {e.response.text}")
            raise # Re-raise the exception after printing

# Parse the model's response to extract score
def parse_score(answer):
    # Attempt to find a number 1, 2, or 3 in the response
    for n in [1, 2, 3]:
        if str(n) in answer:
            return n
    return None

# Run multi-agent evaluation
async def multi_agent_evaluation():
    # Initialize answers
    last_answers = {model: evaluation_question for model in MODELS}

    for round_number in range(1, MAX_ROUNDS + 1):
        print(f"\n=== Round {round_number} ===")
        current_answers = {}

        # Ask each model
        for model in MODELS:

            if round_number == 1:
                prompt = prompt = (
                    f"{evaluation_question}"
                    f"{final_question}"
                )
            else:
                # Build prompt with previous answers of all models
                previous_info = "\n".join(
                    f"{m} answered: {last_answers[m]}" for m in MODELS if m != model
                )
                prompt = (
                    f"{evaluation_question}"
                    f"{final_question}"
                    f"در اینجا جواب مدل های دیگر را می بینید**:\n{previous_info}\n\n"
                    f"جواب قبلی خودت در مورد این جمله**: {last_answers[model]}\n"
                    "در صورتی که لازم می بینید با توجه به نمرات دور قبل امتیاز خود را بهبود دهید **"
                )
            # print('-----prompt: ', prompt, '-------------')
            answer = await ask_model(model, prompt)
            score = parse_score(answer)
            current_answers[model] = {"score": score, "answer": answer}
            print(f"{model} -> Score: {score}, Answer: {answer}")

        # Check if all scores are the same
        scores = [current_answers[m]["score"] for m in MODELS]
        if None not in scores and len(set(scores)) == 1:
            print("\nAll models converged on the same score! Stopping early.")

            # Count the occurrences of each number
            counts = Counter(scores)

            majority_number = counts.most_common(1)[0][0]

            print(f"The majority number is: {majority_number}")
            return majority_number, False

            break

        # Prepare for next round
        last_answers = {m: current_answers[m]["answer"] for m in MODELS}

    # Count the occurrences of each number
    counts = Counter(scores)

    majority_number = counts.most_common(1)[0][0]

    print(f"The majority number is: {majority_number}")
    return majority_number, True
# Run the async function
num, maj = await multi_agent_evaluation()


=== Round 1 ===
openai/gpt-4.1-mini -> Score: 2, Answer: score: 2
reason: جمله از تمثیل آتش برای عشق استفاده کرده که نسبتاً خلاقانه است اما ترکیب مفاهیم "نقاب ها" و "واقعیت" کمتر نوآورانه است و در ادبیات فارسی رایج است.
meta-llama/llama-4-maverick -> Score: 2, Answer: score: 2
reason: جمله تا حدودی خلاقانه است اما استفاده از استعاره "آتش عشق" و عبارت "نقاب از روی واقعیت برداشتن" تا حدی کلیشه‌ای است.
openai/gpt-3.5-turbo -> Score: 2, Answer: score: 2
reason: این جمله نسبتاً خلاقانه است اما مفهوم آتش در دل و نقاب برداشتن از روی واقعیت قابلیت ارتباط با کلیشه‌های رایج را دارد.

All models converged on the same score! Stopping early.
The majority number is: 2


In [ ]:


# Example list of numbers (replace with your list)
numbers = [1, 2, 3, 2, 1, 2, 3, 2, 1, 1, 2, 2, 3, 3, 3]



# If there's a tie for the majority, this will only return one.
# If you need to handle ties, you can check the counts of the top items.

The list of numbers is: [1, 2, 3, 2, 1, 2, 3, 2, 1, 1, 2, 2, 3, 3, 3]
The counts of each number are: Counter({2: 6, 3: 5, 1: 4})
The majority number is: 2


## Log and print

In [ ]:
import sys
import io
import httpx
import asyncio
from collections import Counter

# Create a class to log stdout to a file
class Logger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# Redirect stdout to the logger
log_filename = "evaluation_log_persian.txt" # Using a different filename for this cell
sys.stdout = Logger(log_filename)

# تنظیمات
OPENROUTER_API_KEY = ..
MODELS = ["openai/gpt-4.1-mini", "meta-llama/llama-4-maverick", "openai/gpt-3.5-turbo"]  # Example model names
MAX_ROUNDS = 5

# لیست جملات
sentences = [
    "گربه‌ی خاکستری سریع از روی دیوار پرید.",
    "خورشید در آسمان آبی با زیبایی می‌درخشید.",
    # بقیه 100 جمله
]

# قالب سوال
def get_prompt(sentence, previous_answers=None):
    previous_info = ""
    if previous_answers:
        previous_info = "\n".join(
            f"{model} answered: {previous_answers[model]}" for model in previous_answers
        )

    prompt = f"""
شما باید خلاقیت یک جمله را بر اساس معیار زیر ارزیابی کنید.
لطفاً یک عدد بین **1 تا 3** بدهید. **فقط عدد را بدهید و هیچ توضیحی اضافه نکنید.
کمترین نمره عدد 1 و بیشترین نمره عدد 3 است**

۱. **آیا جمله خلاقانه است و از کلیشه های رایج در زبان فارسی فاصله دارد؟**
- ۱: جمله کاملاً کلیشه‌ای و بدون نوآوری است.
- ۲: جمله نسبتاً خلاقانه است اما هنوز مشابه جملات رایج است.
- ۳: جمله نوآورانه و غیرکلیشه‌ای است.

خلاقیت: X

### حالا جمله زیر را ارزیابی کنید:
جمله: "{sentence}"

{previous_info}
"""
    return prompt

# تماس با OpenRouter
async def ask_model(model, prompt):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }
    async with httpx.AsyncClient(timeout=httpx.Timeout(60.0, connect=60.0)) as client: # Increased timeout
        try:
            response = await client.post(url, headers=headers, json=data)
            response.raise_for_status()
            answer = response.json()["choices"][0]["message"]["content"]
            return answer.strip()
        except httpx.HTTPStatusError as e:
            print(f"HTTP error occurred: {e}")
            print(f"Response body: {e.response.text}")
            raise # Re-raise the exception after printing
        except httpx.ReadTimeout as e:
            print(f"Read timeout occurred: {e}")
            raise # Re-raise the exception after printing


def parse_score(answer):
    for n in ["1", "2", "3"]:
        if n in answer:
            return int(n)
    return None

# ارزیابی چندعامله برای یک جمله
async def evaluate_sentence(sentence):
    last_answers = {model: get_prompt(sentence) for model in MODELS}

    for round_number in range(1, MAX_ROUNDS + 1):
        current_answers = {}

        for model in MODELS:
            previous_others = {m: last_answers[m] for m in MODELS if m != model}
            prompt = get_prompt(sentence, previous_answers=previous_others)

            answer = await ask_model(model, prompt)

            score = parse_score(answer)
            current_answers[model] = {"score": score, "answer": answer}

        scores = [current_answers[m]["score"] for m in MODELS]
        # اگر همه مدل‌ها به یک نمره رسیدند، همین نمره را برگردان
        if None not in scores and len(set(scores)) == 1:
            return scores[0]

        last_answers = {m: current_answers[m]["answer"] for m in MODELS}

    # اگر اجماع نبود، نمره اکثریت را برگردان
    valid_scores = [score for score in scores if score is not None]
    if not valid_scores:  # If no valid scores were returned
        return None
    counter = Counter(valid_scores)
    most_common = counter.most_common()
    if len(most_common) == 1:
        return most_common[0][0]
    else:
        # اگر تساوی بود، نمره بالاتر را انتخاب می‌کنیم
        max_count = most_common[0][1]
        tied_scores = [score for score, count in most_common if count == max_count]
        return max(tied_scores)

# اجرای کل ارزیابی برای همه جملات
async def evaluate_all(sentences):
    sentence_scores = {}
    for idx, sentence in enumerate(sentences, 1):
        print(f"Evaluating sentence {idx}/{len(sentences)}")
        score = await evaluate_sentence(sentence)
        sentence_scores[sentence] = score
    return sentence_scores

# اجرای کد
final_scores = await evaluate_all(sentences)

# نمایش نمرات نهایی هر جمله
for sentence, score in final_scores.items():
    print(f"Sentence: {sentence} -> Score: {score}")

# Restore original stdout (optional, but good practice if you want to stop logging)
# sys.stdout = sys.__stdout__

HTTPStatusError: Client error '403 Forbidden' for url 'https://openrouter.ai/api/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

In [ ]:
import httpx
import asyncio
from collections import Counter

# تنظیمات
OPENROUTER_API_KEY = ...
MODELS = ["openai/gpt-4.1-mini", "meta-llama/llama-4-maverick", "openai/gpt-3.5-turbo"]  # Example model names
MAX_ROUNDS = 5

# لیست جملات
sentences = [
    "گربه‌ی خاکستری سریع از روی دیوار پرید.",
    "خورشید در آسمان آبی با زیبایی می‌درخشید.",
    # بقیه 100 جمله
]

# قالب سوال
def get_prompt(sentence, previous_answers=None):
    previous_info = ""
    if previous_answers:
        previous_info = "\n".join(
            f"{model} answered: {previous_answers[model]}" for model in previous_answers
        )

    prompt = f"""
شما باید خلاقیت یک جمله را بر اساس معیار زیر ارزیابی کنید.
لطفاً یک عدد بین **1 تا 3** بدهید. **فقط عدد را بدهید و هیچ توضیحی اضافه نکنید.
کمترین نمره عدد 1 و بیشترین نمره عدد 3 است**

۱. **آیا جمله خلاقانه است و از کلیشه های رایج در زبان فارسی فاصله دارد؟**
- ۱: جمله کاملاً کلیشه‌ای و بدون نوآوری است.
- ۲: جمله نسبتاً خلاقانه است اما هنوز مشابه جملات رایج است.
- ۳: جمله نوآورانه و غیرکلیشه‌ای است.

خلاقیت: X

### حالا جمله زیر را ارزیابی کنید:
جمله: "{sentence}"

{previous_info}
"""
    return prompt

# تماس با OpenRouter
async def ask_model(model, prompt):
    url = {"https://openrouter.ai/api/v1/chat/completions"}
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    data = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }
    async with httpx.AsyncClient() as client:
        response = await client.post(url, headers=headers, json=data)
        response.raise_for_status()
        answer = response.json()["choices"][0]["message"]["content"]
        return answer.strip()

def parse_score(answer):
    for n in ["1", "2", "3"]:
        if n in answer:
            return int(n)
    return None

# ارزیابی چندعامله برای یک جمله
async def evaluate_sentence(sentence):
    last_answers = {model: get_prompt(sentence) for model in MODELS}

    for round_number in range(1, MAX_ROUNDS + 1):
        current_answers = {}

        for model in MODELS:
            previous_others = {m: last_answers[m] for m in MODELS if m != model}
            prompt = get_prompt(sentence, previous_answers=previous_others)

            answer = await ask_model(model, prompt)

            score = parse_score(answer)
            current_answers[model] = {"score": score, "answer": answer}

        scores = [current_answers[m]["score"] for m in MODELS]
        # اگر همه مدل‌ها به یک نمره رسیدند، همین نمره را برگردان
        if None not in scores and len(set(scores)) == 1:
            return scores[0]

        last_answers = {m: current_answers[m]["answer"] for m in MODELS}

    # اگر اجماع نبود، نمره اکثریت را برگردان
    valid_scores = [score for score in scores if score is not None]
    if not valid_scores:  # If no valid scores were returned
        return None
    counter = Counter(valid_scores)
    most_common = counter.most_common()
    if len(most_common) == 1:
        return most_common[0][0]
    else:
        # اگر تساوی بود، نمره بالاتر را انتخاب می‌کنیم
        max_count = most_common[0][1]
        tied_scores = [score for score, count in most_common if count == max_count]
        return max(tied_scores)

# اجرای کل ارزیابی برای همه جملات
async def evaluate_all(sentences):
    sentence_scores = {}
    for idx, sentence in enumerate(sentences, 1):
        print(f"Evaluating sentence {idx}/{len(sentences)}")
        score = await evaluate_sentence(sentence)
        sentence_scores[sentence] = score
    return sentence_scores

# اجرای کد
final_scores = await evaluate_all(sentences)

# نمایش نمرات نهایی هر جمله
for sentence, score in final_scores.items():
    print(f"Sentence: {sentence} -> Score: {score}")

Evaluating sentence 1/2


ReadTimeout: 